In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from scipy import stats
from scipy.stats import pointbiserialr, spearmanr, pearsonr, chi2_contingency, f_oneway, kruskal

from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
from matplotlib.gridspec import GridSpec
import matplotlib.patches as patches
warnings.filterwarnings('ignore')

In [ ]:
with open('react_ot_vs_dfm_features.pickle', 'rb') as f:
    data = pickle.load(f)

In [ ]:
errors_reactot = {
    'bond_breaking_error': [],
    'bond_formation_error': [],
    'reaction_center_error': [],
    'mean_angle_error': [],
    'mean_torsional_error': [],
    'mean_improper_error': [],
    'max_angle_error': [],
    'max_torsional_error': [],
    'max_improper_error': [],
}

errors_tsdfm = {
    'bond_breaking_error': [],
    'bond_formation_error': [],
    'reaction_center_error': [],
    'mean_angle_error': [],
    'mean_torsional_error': [],
    'mean_improper_error': [],
    'max_angle_error': [],
    'max_torsional_error': [],
    'max_improper_error': [],
}

data_react_ot = data['react_ot_features']
data_ts_dfm = data['ts_dfm_features']

for i in range(len(data_react_ot)):
    errors_reactot['bond_breaking_error'].append(data_react_ot[i]['errors']['bond_errors'][0])
    errors_reactot['bond_formation_error'].append(data_react_ot[i]['errors']['bond_errors'][1])
    errors_reactot['reaction_center_error'].append(data_react_ot[i]['errors']['reaction_center_errors'])
    errors_reactot['mean_angle_error'].append(np.mean(data_react_ot[i]['errors']['angle_errors']))
    errors_reactot['max_angle_error'].append(np.max(data_react_ot[i]['errors']['angle_errors']))
    errors_reactot['mean_torsional_error'].append(np.mean(data_react_ot[i]['errors']['torsional_errors']))
    errors_reactot['max_torsional_error'].append(np.max(data_react_ot[i]['errors']['torsional_errors']))
    errors_reactot['mean_improper_error'].append(np.mean(data_react_ot[i]['errors']['improper_errors']))
    errors_reactot['max_improper_error'].append(np.max(data_react_ot[i]['errors']['improper_errors']))

for i in range(len(data_ts_dfm)):
    errors_tsdfm['bond_breaking_error'].append(data_ts_dfm[i]['errors']['bond_errors'][0])
    errors_tsdfm['bond_formation_error'].append(data_ts_dfm[i]['errors']['bond_errors'][1])
    errors_tsdfm['reaction_center_error'].append(data_ts_dfm[i]['errors']['reaction_center_errors'])
    errors_tsdfm['mean_angle_error'].append(np.mean(data_ts_dfm[i]['errors']['angle_errors']))
    errors_tsdfm['max_angle_error'].append(np.max(data_ts_dfm[i]['errors']['angle_errors']))
    errors_tsdfm['mean_torsional_error'].append(np.mean(data_ts_dfm[i]['errors']['torsional_errors']))
    errors_tsdfm['max_torsional_error'].append(np.max(data_ts_dfm[i]['errors']['torsional_errors']))
    errors_tsdfm['mean_improper_error'].append(np.mean(data_ts_dfm[i]['errors']['improper_errors']))
    errors_tsdfm['max_improper_error'].append(np.max(data_ts_dfm[i]['errors']['improper_errors']))

In [ ]:
error_types = {
    'bond_breaking_error': 'discrete',
    'bond_formation_error': 'discrete',
    'reaction_center_error': 'discrete',
    'max_angle_error': 'continuous',
    'max_torsional_error': 'continuous',
    'max_improper_error': 'continuous'
}

In [ ]:
errors_reactot = pd.DataFrame(errors_reactot)

In [ ]:
errors_tsdfm = pd.DataFrame(errors_tsdfm)

In [ ]:
matplotlib.rcParams.update({'font.size': 8})
matplotlib.rcParams.update({'axes.titlesize': 6})
matplotlib.rcParams.update({'axes.labelsize': 6})
matplotlib.rcParams.update({'legend.fontsize': 6})
matplotlib.rcParams.update({'xtick.labelsize': 6})
matplotlib.rcParams.update({'ytick.labelsize': 6})
matplotlib.rcParams.update({'axes.linewidth': 1.0, 'xtick.major.width': 0.8, 'ytick.major.width': 0.8, 'xtick.minor.width': 0.6, 'ytick.minor.width': 0.6, 'grid.linewidth': 0.5})
matplotlib.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': 'Arial'})
# matplotlib.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': 'Times New Roman'})

In [ ]:
replace_dict_errors = {
        'bond_breaking_error': 'Error Number of Bond Breaking',
        'bond_formation_error': 'Error Number of Bond Formation',
        'reaction_center_error': 'Error Number of Reaction Center',
        'mean_angle_error': 'Mean Angle Error',
        'max_angle_error': 'Max Angle Error',
        'mean_torsional_error': 'Mean Torsional Error',
        'max_torsional_error': 'Max Torsional Error',
        'mean_improper_error': 'Mean Improper Error',
        'max_improper_error': 'Max Improper Error'
}

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from collections import Counter

def plot_combined_comparison(results1, results2, error_types, 
                            method1_name="Method 1", method2_name="Method 2"):
    
    discrete_errors = {k: v for k, v in error_types.items() if v == 'discrete'}
    continuous_errors = {k: v for k, v in error_types.items() if v == 'continuous'}
    
    n_discrete = len(discrete_errors)
    n_continuous = len(continuous_errors)
    
    n_cols = 3 
    n_rows_discrete = 1
    n_rows_continuous = (n_continuous + n_cols - 1) // n_cols
    n_rows_total = n_rows_discrete + n_rows_continuous
    
    width = 180 / 25.4
    height_single = 50 / 25.4
    
    fig = plt.figure(figsize=(width, height_single * n_rows_total))
    
    color1 = '#FF6B6B'
    color2 = '#4ECDC4'
    
    for idx, (error_name, error_type) in enumerate(discrete_errors.items()):
        ax = plt.subplot(n_rows_total, n_cols, idx + 1)
        
        data1 = results1[error_name]
        data2 = results2[error_name]
        
        display_name = replace_dict_errors[error_name]
        
        counter1 = Counter(data1)
        counter2 = Counter(data2)
        
        all_values = sorted(set(list(counter1.keys()) + list(counter2.keys())))
        
        total1 = len(data1)
        total2 = len(data2)
        freq1 = [counter1.get(val, 0) / total1 * 100 for val in all_values]
        freq2 = [counter2.get(val, 0) / total2 * 100 for val in all_values]
        
        x = np.arange(len(all_values))
        width_bar = 0.35
        
        bars1 = ax.bar(x - width_bar/2, freq1, width_bar, 
                      color=color1, alpha=0.8, label=method1_name,
                      edgecolor='black', linewidth=1.5)
        bars2 = ax.bar(x + width_bar/2, freq2, width_bar,
                      color=color2, alpha=0.8, label=method2_name,
                      edgecolor='black', linewidth=1.5)
        
        ax.set_xticks(x)

        if all(isinstance(val, (int, float)) for val in all_values):
            ax.set_xticklabels([str(val) for val in all_values])
        else:
            ax.set_xticklabels(all_values)
        
        ax.set_ylabel('Frequency (%)')
        ax.set_title(f'{display_name}')
        
        max_freq = max(max(freq1), max(freq2))
        ax.set_ylim(0, max_freq * 1.2)
        
        ax.legend()
        
        ax.yaxis.grid(True, linestyle='--', alpha=0.7)
        
        
        for bar in bars1:
            height = bar.get_height()
            # if height > 0: 
            ax.text(bar.get_x() + 0.4 * bar.get_width(), height + 0.5,
                       f'{height:.1f}%', ha='center', va='bottom', size=5)
        
        for bar in bars2:
            height = bar.get_height()
            # if height > 0:
            ax.text(bar.get_x() + 0.6 * bar.get_width(), height + 0.5,
                       f'{height:.1f}%', ha='center', va='bottom', size=5)
    
    for idx, (error_name, error_type) in enumerate(continuous_errors.items()):
        ax = plt.subplot(n_rows_total, n_cols, n_cols + idx + 1)
        
        data1 = results1[error_name]
        data2 = results2[error_name]
        
        display_name = replace_dict_errors[error_name]
        
        sns.kdeplot(data1, ax=ax, label=method1_name, color=color1, 
                   fill=True, alpha=0.6, linewidth=2.)
        sns.kdeplot(data2, ax=ax, label=method2_name, color=color2, 
                   fill=True, alpha=0.6, linewidth=2.)
        
        median1, median2 = np.median(data1), np.median(data2)
        
        ax.axvline(median1, color=color1, linestyle='--', alpha=0.8, 
                  linewidth=2, label=f'{method1_name} Median: {median1:.2f}')
        ax.axvline(median2, color=color2, linestyle='--', alpha=0.8, 
                  linewidth=2, label=f'{method2_name} Median: {median2:.2f}')
        
        ax.set_xlabel('Error (Degree)')
        ax.set_ylabel('Density')
        
        current_xlim = ax.get_xlim()

        ax.set_xlim(left=0, right=max(current_xlim[1], max(data1.max(), data2.max()) * 1.1))
        
        ax.set_title(f'{display_name}')
        
        ax.legend()
        
        ax.grid(True, linestyle='--', alpha=0.3)
        
    
    plt.tight_layout()
    
    
    return fig

In [ ]:
fig = plot_combined_comparison(errors_reactot, errors_tsdfm, error_types,
                              "React-OT", "TS-DFM")

plt.show()
fig.savefig('method_comparison_combined.pdf', dpi=1200, bbox_inches='tight')